# TinyBanglaBERT – ablation + multi-seed (Kaggle) v2
GPU on, Internet on. Add inputs: (1) the dataset with the NEWEST `tiny_bert-fixed.zip`, and (2) the output of your previous notebook run (so the same teacher, 83.13%, is reused).
All 4 rows use the **same architecture and hyper-parameters**; only init (random vs pretrained) and distillation (off/on) differ.
(The unlabeled pool is empty in your data, so it is not part of this ablation.)

In [ ]:
import glob, os, shutil, json, subprocess
# pick ONLY a code folder that has the NEW code (small_random). The previous notebook's output also contains
# an old tiny_bert-main, which must not be used.
cands = []
for h in glob.glob('/kaggle/input/**/tiny_bert-main', recursive=True):
    t = os.path.join(h, 'scripts', 'train.py')
    if os.path.exists(t) and 'small_random' in open(t).read():
        cands.append(h)
assert cands, "No tiny_bert-main with the NEW code found. Upload the newest tiny_bert-fixed.zip as a new dataset version and add it as input."
print("Using code from:", cands[0])
shutil.rmtree('/kaggle/working/tiny_bert-main', ignore_errors=True); shutil.copytree(cands[0], '/kaggle/working/tiny_bert-main')
os.chdir('/kaggle/working/tiny_bert-main'); os.makedirs('experiments', exist_ok=True); os.makedirs('data/processed', exist_ok=True)

In [ ]:
!pip install -q omegaconf lime shap onnx onnxruntime onnxscript optuna statsmodels tabulate

## Teacher + splits: reuse from the previous run if available, otherwise recreate

In [ ]:
prev = glob.glob('/kaggle/input/**/experiments/teacher/model.safetensors', recursive=True)
if prev:
    root = os.path.dirname(os.path.dirname(os.path.dirname(prev[0])))
    shutil.copytree(os.path.dirname(prev[0]), 'experiments/teacher', dirs_exist_ok=True)
    sp = glob.glob('/kaggle/input/**/data/processed/splits.json', recursive=True)
    if sp: shutil.copy(sp[0], 'data/processed/splits.json')
    print('Reused previous teacher from', root)
if not os.path.exists('data/processed/splits.json'):
    !python -m scripts.prepare_data --config configs/base.yaml --seed 42
if not os.path.exists('experiments/teacher/model.safetensors'):
    !python -m scripts.train --config configs/teacher_baseline.yaml --seed 42
m = json.load(open('data/processed/splits.json'))['metadata']; print({k:m[k] for k in ('train_size','val_size','test_size')})
print(open('experiments/teacher/metrics.json').read()[:300])

## Ablation grid × seeds (4 rows)
| Row | init | teacher distillation |
|---|---|---|
| R0 | random weights | no |
| R1 | pretrained (banglabert_small) | no |
| R2 | random weights | yes |
| R3 | pretrained (banglabert_small) | yes  ← proposed model |

`T, A` = best values from your grid search (temperature 1, alpha 0.1).

In [ ]:
T, A = 1.0, 0.1          # from your kd_grid_results.json best_params
SEEDS = [42, 43, 44]     # training seeds (data split stays fixed)
ROWS = {  # name: (init, alpha, use_unlabeled)   -- alpha=1.0 means 'no distillation'
 'R0_random_noKD':     ('small_random', 1.0, False),
 'R1_pretrained_noKD': ('small_subset', 1.0, False),
 'R2_random_KD':       ('small_random', A,   False),
 'R3_pretrained_KD':   ('small_subset', A,   False),
}
for row,(init,alpha,unl) in ROWS.items():
    for seed in SEEDS:
        name = f"{row}_s{seed}"
        if os.path.exists(f'experiments/{name}/test_metrics.json'): continue   # resume-safe
        !python -m scripts.train --config configs/student_kd_improved.yaml --seed {seed} --init {init} --alpha {alpha} --temperature {T} --override training.run_name={name} distillation.use_unlabeled={str(unl).lower()}
        if not os.path.exists(f'experiments/{name}/test_metrics.json'):
            print('!!!!!! RUN FAILED:', name, '- scroll up for the error')

## Results: mean ± std over seeds

In [ ]:
import pandas as pd, numpy as np
rows=[]
for row in ROWS:
    for seed in SEEDS:
        p=f'experiments/{row}_s{seed}/test_metrics.json'
        if os.path.exists(p):
            d=json.load(open(p)); rows.append({'row':row,'seed':seed,'acc':100*d['accuracy'],'f1':d['f1_macro'],'f1_cb':d['f1_clickbait']})
df=pd.DataFrame(rows); df.to_csv('experiments/ablation_all_seeds.csv',index=False)
g=df.groupby('row').agg(acc_mean=('acc','mean'),acc_std=('acc','std'),f1_mean=('f1','mean'),f1_std=('f1','std'),n=('seed','count')).round(3)
t=json.load(open('experiments/teacher/metrics.json')); g['pct_of_teacher_acc']=(g.acc_mean/(100*t['accuracy'])*100).round(1)
print('Teacher acc: %.2f  F1: %.4f'%(100*t['accuracy'],t['f1_macro'])); print(g.to_string()); g.to_csv('experiments/ablation_summary_mean_std.csv')

In [ ]:
# parameter counts (one run per architecture)
import torch
from transformers import AutoModelForSequenceClassification as M
for row in ['R0_random_noKD','R3_pretrained_KD']:
    p = f'experiments/{row}_s{SEEDS[0]}'
    if os.path.isdir(p):
        mdl = M.from_pretrained(p); print(row, round(sum(x.numel() for x in mdl.parameters())/1e6, 2), 'M params')
    else:
        print(row, 'missing (training failed)')

In [ ]:
!cd /kaggle/working/tiny_bert-main && zip -qr /kaggle/working/ablation_multiseed.zip experiments data/processed -x "*/checkpoints/*"
!ls -lh /kaggle/working/ablation_multiseed.zip